In [18]:
import os
from pathlib import Path

import polars as pl
import pandas as pd
import numpy as np

In [11]:
data_path = Path('data')

if not os.path.exists(data_path / 'train.csv'):
    !kaggle competitions download -c playground-series-s6e4
    !unzip playground-series-s6e4.zip -d data
    !rm playground-series-s6e4.zip
    !kaggle datasets download miadul/irrigation-water-requirement-prediction-dataset
    !unzip irrigation-water-requirement-prediction-dataset.zip -d data
    !rm irrigation-water-requirement-prediction-dataset.zip

In [20]:
from mllabs.processor import PolarsLoader, PandasConverter, ExprProcessor
from sklearn.pipeline import make_pipeline

loader = make_pipeline(
    PolarsLoader(predefined_types={'id': pl.Int64}),
    PandasConverter(index_col = 'id')
)
df_train = loader.fit_transform([data_path / 'train.csv'])
df_test = loader.transform([data_path / 'test.csv'])

In [22]:
df_org = loader.transform([data_path / 'irrigation_prediction.csv'])
df_org = df_org.set_index(pd.Index(-np.arange(1, len(df_org) + 1), name = 'id'))

In [23]:
df_data_spec = loader[0].df_type_.join(
    df_train.dtypes.rename('actual_dtype')
).pipe(lambda x: x.loc[x['actual_dtype'].notna()])
display(df_data_spec)
df_train.shape

,min,max,na,count,n_unique,dtype,f32,i32,i16,i8,actual_dtype
feature,,,,,,,,,,,
Crop_Growth_Stage,NaN,NaN,0.0,630000.0,4.0,String,False,False,False,False,category
Crop_Type,NaN,NaN,0.0,630000.0,6.0,String,False,False,False,False,category
Electrical_Conductivity,0.10,3.50,0.0,630000.0,341.0,Float64,True,True,True,True,float32
Field_Area_hectare,0.30,15.00,0.0,630000.0,1466.0,Float64,True,True,True,True,float32
Humidity,25.00,94.99,0.0,630000.0,6475.0,Float64,True,True,True,True,float32
Irrigation_Need,NaN,NaN,0.0,630000.0,3.0,String,False,False,False,False,category
Irrigation_Type,NaN,NaN,0.0,630000.0,4.0,String,False,False,False,False,category
Mulching_Used,NaN,NaN,0.0,630000.0,2.0,String,False,False,False,False,category
Organic_Carbon,0.30,1.60,0.0,630000.0,131.0,Float64,True,True,True,True,float32


(630000, 20)

In [28]:
df_data_spec.loc[df_data_spec['actual_dtype'] == 'category'].index

Index(['Crop_Growth_Stage', 'Crop_Type', 'Irrigation_Need', 'Irrigation_Type',
       'Mulching_Used', 'Region', 'Season', 'Soil_Type', 'Water_Source'],
      dtype='object', name='feature')

In [29]:
df_data_spec.loc[df_data_spec['actual_dtype'] == 'float32'].index

Index(['Electrical_Conductivity', 'Field_Area_hectare', 'Humidity',
       'Organic_Carbon', 'Previous_Irrigation_mm', 'Rainfall_mm',
       'Soil_Moisture', 'Soil_pH', 'Sunlight_Hours', 'Temperature_C',
       'Wind_Speed_kmh'],
      dtype='object', name='feature')

In [30]:
y = 'Irrigation_Need'
X_cat = ['Crop_Growth_Stage', 'Crop_Type', 'Irrigation_Type', 'Mulching_Used', 'Region', 'Season', 'Soil_Type', 'Water_Source']
X_num = ['Electrical_Conductivity', 'Field_Area_hectare', 'Humidity', 'Organic_Carbon', 'Previous_Irrigation_mm', 'Rainfall_mm', 
         'Soil_Moisture', 'Soil_pH', 'Sunlight_Hours', 'Temperature_C','Wind_Speed_kmh']

In [35]:
pd.concat([
    df_train[X_cat].apply(lambda x: set(x.unique())).rename('train'),
    df_test[X_cat].apply(lambda x: set(x.unique())).rename('test')
], axis = 1).apply(
    lambda x: x['test'] - x['train'], axis = 1 
)

Crop_Growth_Stage    {}
Crop_Type            {}
Irrigation_Type      {}
Mulching_Used        {}
Region               {}
Season               {}
Soil_Type            {}
Water_Source         {}
dtype: object

In [37]:
df_train[X_cat].nunique().rename('cardinality').to_frame().T

,Crop_Growth_Stage,Crop_Type,Irrigation_Type,Mulching_Used,Region,Season,Soil_Type,Water_Source
cardinality,4,6,4,2,5,3,4,4
